In [3]:
import sys
import os

# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

In [4]:
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema
from preprocess_sets import subPath, participantsInfoPath, processSubPSDs, processSub

In [5]:
# test_imports.py
from preprocess_sets import subPath, participantsInfoPath, processSubPSDs, processSub

# Test 1: Check if functions are imported correctly
print("Function check:")
print("subPath exists:", subPath is not None)
print("participantsInfoPath exists:", participantsInfoPath is not None)
print("processSubPSDs exists:", processSubPSDs is not None)
print("processSub exists:", processSub is not None)

# Test 2: Check if path functions return expected values
print("\nPath check:")
try:
    participants_path = participantsInfoPath()
    print("Participants info path:", participants_path)
    print("Path exists:", os.path.exists(participants_path))
except Exception as e:
    print("Error getting participants path:", str(e))

# Test 3: Check if subPath works for a sample subject
print("\nsubPath check:")
try:
    subject_id = "001"  # Change to a subject ID you know exists
    path = subPath(subject_id, derivatives=True)
    print(f"Path for subject {subject_id}:", path)
    print("Path exists:", os.path.exists(path))
except Exception as e:
    print(f"Error getting path for subject {subject_id}:", str(e))

# Only run this if you're confident the above tests passed
# as this will attempt to actually load data
print("\nMini processSub check:")
try:
    import time
    start = time.time()
    subject_id = "001"  # Use a known subject ID
    print(f"Attempting to get first epoch for {subject_id}...")
    epochs = processSub(subject_id)
    print(f"Got {len(epochs)} epochs in {time.time() - start:.2f} seconds")
    print("First epoch shape:", epochs[0].get_data().shape)
except Exception as e:
    print(f"Error processing subject {subject_id}:", str(e))
print("check end")

Function check:
subPath exists: True
participantsInfoPath exists: True
processSubPSDs exists: True
processSub exists: True

Path check:
Participants info path: /Users/user/eeg-ds004504/ds004504/participants.tsv
Path exists: True

subPath check:
subPath 001
Path handed: /Users/user/eeg-ds004504/ds004504/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
Path for subject 001: /Users/user/eeg-ds004504/ds004504/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
Path exists: True

Mini processSub check:
Attempting to get first epoch for 001...
processSub 001
subPath 001
Path handed: /Users/user/eeg-ds004504/ds004504/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
Got 398 epochs in 2.48 seconds
First epoch shape: (1, 19, 1501)
check end


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
from schema_definition import get_subject_schema, get_feature_schema
from feature_extraction import processEpoch, processSub
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame

In [7]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame


def load_subjects_df(spark: SparkSession, participants_path: str) -> DataFrame:
    """
    Reads participants.tsv and returns a Spark DataFrame
    with columns SubjectID and Group for groups A, C, and F.

    Parameters:
        spark (SparkSession): Active Spark session
        participants_path (str): Path to the participants.tsv file

    Returns:
        Spark DataFrame with SubjectID and Group columns
    """
    participantsInfo = pd.read_table(participants_path)

    records = []
    for group_code in ["A", "C", "F"]:
        group_subjects = participantsInfo[participantsInfo["Group"] == group_code]["participant_id"].tolist()
        for sub in group_subjects:
            records.append((sub, group_code))
    return spark.createDataFrame(records, schema=get_subject_schema())


# put code below in main file 

In [8]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

# Now create a new SparkSession with local binding address
from pyspark.sql import SparkSession
import os

# Set environment variables
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# Create new session with explicit local binding
spark = SparkSession.builder \
    .appName("EEG_Analysis") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") \
    .master("local[*]") \
    .getOrCreate()

print("New Spark session created successfully")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/31 21:13:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


New Spark session created successfully


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 49623)
Traceback (most recent call last):
  File "/usr/local/Cellar/python@3.12/3.12.6/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/local/Cellar/python@3.12/3.12.6/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/usr/local/Cellar/python@3.12/3.12.6/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/local/Cellar/python@3.12/3.12.6/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 761, in __init__
    self.handle()
  File "/Users/user/jupyter-venv/lib/python3.12/site-packages/pyspark/accumul

In [9]:
spark = SparkSession.builder.appName("MyApp").getOrCreate()

subject_df = load_subjects_df(spark, "../ds004504/participants.tsv")

# subjects_df.show()
subject_df.show(n=subject_df.count(), truncate=False)# Optionally, you can save this DataFrame

25/03/31 21:13:58 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
                                                                                

+---------+-----+
|SubjectID|Group|
+---------+-----+
|sub-001  |A    |
|sub-002  |A    |
|sub-003  |A    |
|sub-004  |A    |
|sub-005  |A    |
|sub-006  |A    |
|sub-007  |A    |
|sub-008  |A    |
|sub-009  |A    |
|sub-010  |A    |
|sub-011  |A    |
|sub-012  |A    |
|sub-013  |A    |
|sub-014  |A    |
|sub-015  |A    |
|sub-016  |A    |
|sub-017  |A    |
|sub-018  |A    |
|sub-019  |A    |
|sub-020  |A    |
|sub-021  |A    |
|sub-022  |A    |
|sub-023  |A    |
|sub-024  |A    |
|sub-025  |A    |
|sub-026  |A    |
|sub-027  |A    |
|sub-028  |A    |
|sub-029  |A    |
|sub-030  |A    |
|sub-031  |A    |
|sub-032  |A    |
|sub-033  |A    |
|sub-034  |A    |
|sub-035  |A    |
|sub-036  |A    |
|sub-037  |C    |
|sub-038  |C    |
|sub-039  |C    |
|sub-040  |C    |
|sub-041  |C    |
|sub-042  |C    |
|sub-043  |C    |
|sub-044  |C    |
|sub-045  |C    |
|sub-046  |C    |
|sub-047  |C    |
|sub-048  |C    |
|sub-049  |C    |
|sub-050  |C    |
|sub-051  |C    |
|sub-052  |C    |
|sub-053  

  # end main code

In [10]:

subject_df = load_subjects_df(spark, "../ds004504/participants.tsv")

# subjects_df.show()
subject_df.show(n=subject_df.count(), truncate=False)# Optionally, you can save this DataFrame

+---------+-----+
|SubjectID|Group|
+---------+-----+
|sub-001  |A    |
|sub-002  |A    |
|sub-003  |A    |
|sub-004  |A    |
|sub-005  |A    |
|sub-006  |A    |
|sub-007  |A    |
|sub-008  |A    |
|sub-009  |A    |
|sub-010  |A    |
|sub-011  |A    |
|sub-012  |A    |
|sub-013  |A    |
|sub-014  |A    |
|sub-015  |A    |
|sub-016  |A    |
|sub-017  |A    |
|sub-018  |A    |
|sub-019  |A    |
|sub-020  |A    |
|sub-021  |A    |
|sub-022  |A    |
|sub-023  |A    |
|sub-024  |A    |
|sub-025  |A    |
|sub-026  |A    |
|sub-027  |A    |
|sub-028  |A    |
|sub-029  |A    |
|sub-030  |A    |
|sub-031  |A    |
|sub-032  |A    |
|sub-033  |A    |
|sub-034  |A    |
|sub-035  |A    |
|sub-036  |A    |
|sub-037  |C    |
|sub-038  |C    |
|sub-039  |C    |
|sub-040  |C    |
|sub-041  |C    |
|sub-042  |C    |
|sub-043  |C    |
|sub-044  |C    |
|sub-045  |C    |
|sub-046  |C    |
|sub-047  |C    |
|sub-048  |C    |
|sub-049  |C    |
|sub-050  |C    |
|sub-051  |C    |
|sub-052  |C    |
|sub-053  

In [11]:
subject_df.printSchema()

root
 |-- SubjectID: string (nullable = false)
 |-- Group: string (nullable = false)



In [12]:
subject_df.filter(subject_df.SubjectID == "sub-001").show()


+---------+-----+
|SubjectID|Group|
+---------+-----+
|  sub-001|    A|
+---------+-----+



# Getting Spark Function to work

In [13]:
# Get the SparkContext from your existing SparkSession
sc = spark.sparkContext

# Add your Python modules to all workers
import os

# For Jupyter notebook, use relative path to src directory
script_dir = os.path.abspath(os.path.join(os.getcwd(), "../src"))
print(f"Using source directory: {script_dir}")

# Check if the directory exists
if not os.path.exists(script_dir):
    print(f"Warning: Directory {script_dir} does not exist!")
    # Fallback to alternative paths
    possible_paths = [
        os.path.abspath(os.path.join(os.getcwd(), "src")),
        os.path.abspath(os.path.join(os.getcwd(), "../src")),
        os.path.abspath(os.path.join(os.getcwd(), "../../src"))
    ]
    
    for path in possible_paths:
        if os.path.exists(path):
            print(f"Found alternative path: {path}")
            script_dir = path
            break
    else:
        print("Could not find src directory. Please specify the full path.")

# List files in the directory to verify
print("Files in the directory:")
try:
    for file in os.listdir(script_dir):
        if file.endswith('.py'):
            print(f"  - {file}")
except Exception as e:
    print(f"Error listing directory: {e}")

# Add all necessary modules
try:
    sc.addPyFile(os.path.join(script_dir, "feature_extraction.py"))
    print("Added feature_extraction.py")
    sc.addPyFile(os.path.join(script_dir, "preprocess_sets.py"))
    print("Added preprocess_sets.py")
    sc.addPyFile(os.path.join(script_dir, "schema_definition.py"))
    print("Added schema_definition.py")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Using source directory: /Users/user/eeg-ds004504/src
Files in the directory:
  - populate_schemas.py
  - preprocess_sets.py
  - __init__.py
  - test.py
  - feature_extraction.py
  - schema_definition.py
Added feature_extraction.py
Added preprocess_sets.py
Added schema_definition.py


In [14]:
# debugging funcion

In [15]:
# What do ... 

#possibly adding jobblib but i don't think there is a real need to 

In [16]:
@pandas_udf(get_feature_schema(), PandasUDFType.GROUPED_MAP)
def extract_features_udtf(pdf):
    import time
    from feature_extraction import processEpoch, processSub
    from schema_definition import get_feature_schema, get_subject_schema
    import mne
    print("function ran")
    rows = []
    start = time.time()
    for _, row in pdf.iterrows():
        subject_id = row["SubjectID"]
        try:
            print(f"Processing subject {subject_id}")
            epochs = processSub(subject_id, derivatives=False)
            print(f"Got {len(epochs)} epochs for {subject_id}")
            print(type(epochs)) 
            # for i, epoch in enumerate(epochs):
            for i in range(len(epochs)):
                epoch = epochs[i]
                if i < 2:  # Just print info for the first 2 epochs to avoid spam
                    # print(f"Epoch {i} shape: {epoch.to_data_frame().shape()}")
                    print(f"Epoch {i}")
                    print(type(epoch))
                    print(type(epochs[i]))
                epoch_id = f"ep-{i}"
                features = processEpoch(epoch)
                
                if i < 2:  # Debug output
                    # print(f"Epoch {i} features count: {len(features) if features else 0}")
                    if features and len(features) > 0:
                        pass
                        # print(f"First feature sample: {next(iter(features))}")
                
                for item in features:
                    # Check the structure of each item
                    # print(f"item {item}")
                    electrode_band_key, stats_value = item
                    electrode, band = electrode_band_key
                    # print(f"Adding: {subject_id}, {epoch_id}, {band}, {electrode}, stats: {stats_value}")
                    
                    # Add to results - adjust this based on actual structure 
                    # TODO : **Error here ! have to line it up! :)
                    try:
                        rows.append((subject_id, epoch_id, band, electrode, *stats_value))
                    except Exception as e:
                        print(f"Error appending row: {e}, stats_value: {stats_value}")
            
            print(f"Total rows collected: {len(rows)}")
            
        except Exception as e:
            print(f"Error processing {subject_id}: {e}")
            import traceback
            traceback.print_exc()
            
    # Print final row count before returning
    print(f"Returning DataFrame with {len(rows)} rows")
    
    # Check if we have column names from schema
    schema_fields = get_feature_schema()
    column_names = [f.name for f in schema_fields]
    print(f"Column names from schema: {column_names}")
    #error after here 
    import pandas as pd
    result_df = pd.DataFrame(rows, columns=column_names)
    # print(f"Result DataFrame shape: {result_df.shape}")
    print(time.time() - start)
    return result_df

In [17]:
print(processSub('sub-001', derivatives=False))

processSub sub-001
subPath sub-001
Path handed: /Users/user/eeg-ds004504/ds004504/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
<Epochs | 398 events (all good), 0 – 3 s (baseline off), ~86.6 MiB, data loaded,
 '1': 398>


In [18]:
# testing a single subject

In [19]:
start = time.time()
result = (
    subject_df
    .filter((subject_df.SubjectID == "sub-001") & (subject_df.Group == "A"))
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)
result.show()
print(time.time() - start)

/Users/user/jupyter-venv/lib/python3.12/site-packages/pyspark/sql/pandas/group_ops.py:104: UserWarning: It is preferred to use 'applyInPandas' over this API. This API will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(
function ran                                                        (0 + 1) / 1]
Processing subject sub-001
processSub sub-001
subPath sub-001
Path handed: /Users/user/eeg-ds004504/ds004504/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
Got 398 epochs for sub-001
<class 'mne.epochs.Epochs'>
Epoch 0
<class 'mne.epochs.Epochs'>
<class 'mne.epochs.Epochs'>
Epoch 1
<class 'mne.epochs.Epochs'>
<class 'mne.epochs.Epochs'>


+---------+-------+--------+---------+--------------------+
|SubjectID|EpochID|WaveBand|Electrode|               Power|
+---------+-------+--------+---------+--------------------+
|  sub-001|   ep-0|   Delta|      Fp1| 0.08257892642542036|
|  sub-001|   ep-0|   Theta|      Fp1|0.005611645685714301|
|  sub-001|   ep-0|   Alpha|      Fp1|0.001143220388171...|
|  sub-001|   ep-0|    Beta|      Fp1|1.958040080323614...|
|  sub-001|   ep-0|   Total|      Fp1|0.011235955056179778|
|  sub-001|   ep-0|   Delta|      Fp2| 0.08389220643958978|
|  sub-001|   ep-0|   Theta|      Fp2|0.004672355992737973|
|  sub-001|   ep-0|   Alpha|      Fp2|9.638188967593915E-4|
|  sub-001|   ep-0|    Beta|      Fp2|1.768820461211869...|
|  sub-001|   ep-0|   Total|      Fp2|0.011235955056179771|
|  sub-001|   ep-0|   Delta|       F3| 0.08212274846138011|
|  sub-001|   ep-0|   Theta|       F3|0.005595318986261473|
|  sub-001|   ep-0|   Alpha|       F3|0.001112336388491...|
|  sub-001|   ep-0|    Beta|       F3|2.

Total rows collected: 37810
Returning DataFrame with 37810 rows
Column names from schema: ['SubjectID', 'EpochID', 'WaveBand', 'Electrode', 'Power']
17.640578985214233
                                                                                

In [19]:
 # testing everything 

In [23]:
# Start both groups processing (lazy evaluation means these won't execute yet)
result_group_a = (
    subject_df
    .filter(subject_df.Group == "A")
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)

result_group_c = (
    subject_df
    .filter(subject_df.Group == "C")
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)

# Force execution of both by calling an action on each
# This launches both jobs in parallel
from pyspark.sql import functions as F

# Use persist to cache the results in memory
result_group_a = result_group_a.persist()
result_group_c = result_group_c.persist()

# Trigger execution with actions
count_a = result_group_a.count()
count_c = result_group_c.count()

print(f"Processed {count_a} records for Alzheimer's group")
print(f"Processed {count_c} records for Control group")

/Users/user/jupyter-venv/lib/python3.12/site-packages/pyspark/sql/pandas/group_ops.py:104: UserWarning: It is preferred to use 'applyInPandas' over this API. This API will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(
function ranfunction ran==============================>        (168 + 12) / 200]
function ranfunction ran

function ran
function ran
function ran
function ranfunction ran
Processing subject sub-021Processing subject sub-020

processSubprocessSub  sub-021sub-020

Processing subject sub-012
processSub sub-012
function ran
Processing subject sub-009
processSub sub-009
Processing subject sub-001
processSub sub-001
Processing subject sub-017


processSub sub-017
function ran
Processing subject sub-018
processSub sub-018
Processing subject sub-029
processSub sub-029
Processing subject sub-022
processSub sub-022
Processing subject sub-016
processSub Processing subject sub-002sub-016
processSub sub-002

function ran
Processing subject su

Processed 1856870 records for Alzheimer's group
Processed 1543465 records for Control group


In [25]:
# For Group A
count_a = result_group_a.count()
subjects_a = result_group_a.select("SubjectID").distinct().count()
epochs_a = result_group_a.select("SubjectID", "EpochID").distinct().count()
epochs_per_subject_a = result_group_a.groupBy("SubjectID").agg(
    F.countDistinct("EpochID").alias("EpochCount")
)

# For Group C
count_c = result_group_c.count()
subjects_c = result_group_c.select("SubjectID").distinct().count()
epochs_c = result_group_c.select("SubjectID", "EpochID").distinct().count()
epochs_per_subject_c = result_group_c.groupBy("SubjectID").agg(
    F.countDistinct("EpochID").alias("EpochCount")
)

# Print summary for Group A
print(f"Group A (Alzheimer's):")
print(f"Total records: {count_a}")
print(f"Unique subjects: {subjects_a}")
print(f"Total unique epochs: {epochs_a}")
print(f"Average epochs per subject: {epochs_a/subjects_a:.2f}")

# Print summary for Group C
print(f"Group C (Controls):")
print(f"Total records: {count_c}")
print(f"Unique subjects: {subjects_c}")
print(f"Total unique epochs: {epochs_c}")
print(f"Average epochs per subject: {epochs_c/subjects_c:.2f}")

# If you want to see the epoch count for each subject in Group A
print("\nEpoch counts per subject (Group A):")
epochs_per_subject_a.show()

# If you want to see the epoch count for each subject in Group C
print("\nEpoch counts per subject (Group C):")
epochs_per_subject_c.show()

# If you need a more detailed breakdown including bands and electrodes
detailed_stats_a = result_group_a.groupBy("SubjectID").agg(
    F.countDistinct("EpochID").alias("Epochs"),
    F.countDistinct("WaveBand").alias("WaveBands"),
    F.countDistinct("Electrode").alias("Electrodes"),
    F.count("*").alias("TotalFeatures")
)

detailed_stats_c = result_group_c.groupBy("SubjectID").agg(
    F.countDistinct("EpochID").alias("Epochs"),
    F.countDistinct("WaveBand").alias("WaveBands"),
    F.countDistinct("Electrode").alias("Electrodes"),
    F.count("*").alias("TotalFeatures")
)

print("\nDetailed stats for Group A:")
detailed_stats_a.show()

print("\nDetailed stats for Group C:")
detailed_stats_c.show()

Group A (Alzheimer's):
Total records: 1856870
Unique subjects: 36
Total unique epochs: 19546
Average epochs per subject: 542.94
Group C (Controls):
Total records: 1543465
Unique subjects: 29
Total unique epochs: 16247
Average epochs per subject: 560.24

Epoch counts per subject (Group A):


+---------+----------+
|SubjectID|EpochCount|
+---------+----------+
|  sub-020|       578|
|  sub-033|       470|
|  sub-012|       597|
|  sub-002|       527|
|  sub-011|       513|
|  sub-016|       655|
|  sub-017|       563|
|  sub-025|       464|
|  sub-021|       614|
|  sub-018|       563|
|  sub-029|       492|
|  sub-001|       398|
|  sub-009|       408|
|  sub-022|       548|
|  sub-036|       567|
|  sub-004|       470|
|  sub-027|       552|
|  sub-006|       423|
|  sub-019|       612|
|  sub-030|       370|
+---------+----------+
only showing top 20 rows


Epoch counts per subject (Group C):


+---------+----------+
|SubjectID|EpochCount|
+---------+----------+
|  sub-058|       507|
|  sub-057|       529|
|  sub-037|       517|
|  sub-055|       548|
|  sub-062|       607|
|  sub-060|       499|
|  sub-049|       521|
|  sub-053|       531|
|  sub-059|       525|
|  sub-064|       565|
|  sub-054|       560|
|  sub-039|       570|
|  sub-065|       588|
|  sub-063|       538|
|  sub-061|       537|
|  sub-050|       550|
|  sub-041|       590|
|  sub-052|       506|
|  sub-046|       504|
|  sub-045|       574|
+---------+----------+
only showing top 20 rows


Detailed stats for Group A:


+---------+------+---------+----------+-------------+
|SubjectID|Epochs|WaveBands|Electrodes|TotalFeatures|
+---------+------+---------+----------+-------------+
|  sub-020|   578|        5|        19|        54910|
|  sub-033|   470|        5|        19|        44650|
|  sub-012|   597|        5|        19|        56715|
|  sub-002|   527|        5|        19|        50065|
|  sub-011|   513|        5|        19|        48735|
|  sub-016|   655|        5|        19|        62225|
|  sub-017|   563|        5|        19|        53485|
|  sub-025|   464|        5|        19|        44080|
|  sub-021|   614|        5|        19|        58330|
|  sub-018|   563|        5|        19|        53485|
|  sub-029|   492|        5|        19|        46740|
|  sub-001|   398|        5|        19|        37810|
|  sub-009|   408|        5|        19|        38760|
|  sub-022|   548|        5|        19|        52060|
|  sub-036|   567|        5|        19|        53865|
|  sub-004|   470|        5|

[Stage 111:====================================================>(197 + 3) / 200]

+---------+------+---------+----------+-------------+
|SubjectID|Epochs|WaveBands|Electrodes|TotalFeatures|
+---------+------+---------+----------+-------------+
|  sub-058|   507|        5|        19|        48165|
|  sub-057|   529|        5|        19|        50255|
|  sub-037|   517|        5|        19|        49115|
|  sub-055|   548|        5|        19|        52060|
|  sub-062|   607|        5|        19|        57665|
|  sub-060|   499|        5|        19|        47405|
|  sub-049|   521|        5|        19|        49495|
|  sub-053|   531|        5|        19|        50445|
|  sub-059|   525|        5|        19|        49875|
|  sub-064|   565|        5|        19|        53675|
|  sub-054|   560|        5|        19|        53200|
|  sub-039|   570|        5|        19|        54150|
|  sub-063|   538|        5|        19|        51110|
|  sub-065|   588|        5|        19|        55860|
|  sub-061|   537|        5|        19|        51015|
|  sub-050|   550|        5|

In [26]:
# Get the counts first
count_a = result_group_a.count()
count_c = result_group_c.count()
subjects_a = result_group_a.select("SubjectID").distinct().count()
subjects_c = result_group_c.select("SubjectID").distinct().count()

print(f"Group A: {count_a} records, {subjects_a} subjects")
print(f"Group C: {count_c} records, {subjects_c} subjects")

# Convert to pandas
df_a = result_group_a.toPandas()
df_c = result_group_c.toPandas()

# Save to pickle files for handoff to your ML module
df_a.to_pickle("eeg_features_alzheimers.pkl")
df_c.to_pickle("eeg_features_controls.pkl")



print(f"Saved pandas DataFrames to pickle files")
print(f"Alzheimer's group shape: {df_a.shape}")
print(f"Control group shape: {df_c.shape}")

Group A: 1856870 records, 36 subjects
Group C: 1543465 records, 29 subjects


Saved pandas DataFrames to pickle files
Alzheimer's group shape: (1856870, 5)
Control group shape: (1543465, 5)


In [22]:
subject_df_pd = subject_df.toPandas()
subject_df_pd.to_pickle("subject_df.pkl")
print(f"Saved pandas DataFrames to pickle files")
print(f"Alzheimer's group shape: {subject_df_pd.shape}")

Saved pandas DataFrames to pickle files
Alzheimer's group shape: (88, 2)
